In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
import lingam
import networkx as nx
import matplotlib.pyplot as plt

#set project root as .../recidivism-causal
project_root = Path("/dcs/23/u2200504/thesis/recidivism-causal").resolve()

#set causal pitfalls root
cp_root = project_root / "data" / "raw" / "CausalPitfallsData"
#test it works
cp_root

#path to put results
output_dir = project_root/"results"/"graphs_LiM"
output_dir.mkdir(parents=True,exist_ok=True)

In [3]:
"""LiM expects a numeric matrix with continuous and discrete columns as well as
an array per variable indicating 0 for discrete and 1 for continuous.
Similar to what we did for DAGBagM, we write a helper function to infer the datatype."""

def infer_type(df):
    """
    Infers a 'flag array' for a given dataframe.
    0 corresponds to discrete variables, 1 to continuous variables
    """

    flags=[]
    for col in df.columns:
        x = df[col].dropna()
        #classify numeric columns
        if np.issubdtype(x.dtype, np.number):
            vals = x.unique()
            if len(vals) <= 1:
                flags.append(0) #variable is discrete
            else:
                flags.append(1) # variable is continuous
    return np.asarray(flags, dtype=int)

def run_lim(df, seed=1):
    """
    Clean dataframe and generate the numeric matrix as well as flag array.
    Run LiM on the data.
    """

    df_clean=df.dropna().copy()
    flags = infer_type(df_clean)
    flags_2d = flags.reshape(1,-1)
    cols = list(df_clean.columns)

    X = df_clean.to_numpy(dtype=float)

    model = lingam.LiM()

    model.fit(X, flags_2d, only_global=True)
    #LiM creates an adjacency matrix
    M = model.adjacency_matrix_
    #coerce adjacency matrix into a matrix of 0's and 1's
    adj = (np.abs(M) > 0).astype(int)

    return adj, cols

In [4]:
def draw_graph(adj, nodes, output_path):
    G = nx.DiGraph()
    #add nodes
    G.add_nodes_from(nodes)
    
    #add directed edges, adj[i,j] = 1 => i -> j
    for i, src in enumerate(nodes):
        for j, tgt in enumerate(nodes):
            if adj[i, j] == 1:
                G.add_edge(src, tgt)

    plt.figure(figsize=(10,8))
    pos = nx.spring_layout(G, k=1.2, iterations=500, seed=0)
    nx.draw(G, pos, with_labels=True, labels={node: node for node in nodes},
            node_size=900, font_size=8, arrowsize=10)
    plt.savefig(out_path, dpi=150)
    plt.close()

In [5]:
from graphviz import Digraph
from pathlib import Path
import numpy as np

def draw_graphviz_dag(adj, out_path, node_labels=None, engine="dot"):

    adj = np.asarray(adj)
    if adj.ndim == 1:
        adj = adj.reshape(1, 1)

    n = adj.shape[0]

    # Default labels
    if node_labels is None:
        node_labels = [f"X{i}" for i in range(n)]
    else:
        node_labels = list(node_labels)[:n]

    out_path = Path(out_path)
    g = Digraph(format="png", engine=engine)

    # Thesis‑friendly black & white style
    g.attr(rankdir="TB")  # top‑to‑bottom; use "LR" if you prefer left‑to‑right
    g.attr(
        "node",
        shape="ellipse",
        style="solid",
        color="black",
        fontname="Helvetica",   # or "Times New Roman" / "Palatino"
        fontsize="10",
    )
    g.attr(
        "edge",
        color="black",
        arrowsize="0.7",
    )

    # Add nodes
    for name in node_labels:
        g.node(name, label=name)

    # Add edges for weights above threshold
    for i, src in enumerate(node_labels):
        for j, tgt in enumerate(node_labels):
            w = adj[i, j]
            if abs(w) > 0:
                g.edge(src, tgt)

    out_path.parent.mkdir(parents=True, exist_ok=True)
    g.render(filename=out_path.with_suffix("").as_posix(), cleanup=True)


In [6]:
def one_hot_encode(df):
    non_numeric_cols = df.select_dtypes(exclude=["number"]).columns

    #return the same df if no non numeric
    if len(non_numeric_cols) == 0:
        return df.copy()

    df_encoded = pd.get_dummies(
        df,
        columns=list(non_numeric_cols),
        drop_first=False, 
        dtype=int,
    )
    return df_encoded

In [7]:
#scoring utility function given two adjacency matricies
def score_graph(W_est, W_true, labels_est, labels_true):
    #number of variables
    p = W_est.shape[0]

    #node orderings
    labels_est = list(labels_est)
    labels_true = list(labels_true)
    common = sorted(set(labels_est) & set(labels_true))

    #map ids
    idx_est = [labels_est.index(v) for v in common]
    idx_true = [labels_true.index(v) for v in common]

    W_est_aligned = np.asarray(W_est)[np.ix_(idx_est, idx_est)]
    W_true_aligned = np.asarray(W_true)[np.ix_(idx_true, idx_true)]

    est = (W_est!=0).astype(int)
    true = (W_true !=0).astype(int)

    #true positive, false positive, false negative, true negative
    tp = np.sum((est==1) & (true==1))
    fp = np.sum((est==1) & (true == 0))
    fn = np.sum((est==0) & (true == 1))
    tn = np.sum((est==0) & (true == 0))

    #structural hamming distance, true positive rate, false positive rate
    shd = fp + fn
    tpr = tp/(tp+fn) if (tp+fn) > 0 else np.nan
    fpr = fp/(fp+tn) if (fp + tn)>0 else np.nan
    fdr = fp/(tp+fp) if (tp+fp)>0 else np.nan

    #store scores in dictionary format for each 'experiment'
    return dict(TP=int(tp), FP=int(fp), FN=int(fn), TN=int(tn), SHD=int(shd), TPR = tpr, FPR = fpr, FDR= fdr)

In [8]:
csv_files = sorted(cp_root.rglob("*.csv"))
results = []

for csv_path in csv_files:
    #relative path
    rel = csv_path.relative_to(cp_root)

    #we only examine those with known ground truths i.e with _truth
    if csv_path.stem.endswith("_truth"):
        continue

    #expected ground truth path
    truth_path = csv_path.with_name(csv_path.stem + "_truth.csv")
    if not truth_path.exists():
        print("  Skipped (no ground-truth file)")
        continue
        
    print(f"Processing: {rel}")

    try:
        df = pd.read_csv(csv_path)
        if df.empty:
            print("  Skipped (empty file)")
            continue
        df_enc = one_hot_encode(df)
        
        # Run LiM on this dataset
        adj,nodes = run_lim(df_enc, seed=1)

        #Output schema scenario__file__LiM.png
        parts = rel.parts           
        scenario = parts[0] if len(parts) > 1 else "root"
        name_no_ext = csv_path.stem

        out_name = f"{scenario}__{name_no_ext}__LiM.png"
        out_path = output_dir / out_name

        # Save PNG
        #draw_graph(adj, nodes, out_path)
        draw_graphviz_dag(adj, out_path, nodes)
        print(f"  Saved graph to {out_path.relative_to(project_root)}")

        #load ground truth and score
        gt_df = pd.read_csv(truth_path)
        gt_df.index.name = None
        gt_df.columns.name = None

        W_true = gt_df.to_numpy()
            
        # compute scores
        metrics = score_graph(adj, W_true, labels_est=list(df.columns), labels_true=list(gt_df.columns))
        print("metrics computed")
        print(metrics)
        metrics.update(
            dict(
                scenario=scenario,
                dataset=name_no_ext,
                algo="lim"
            )
        )
        results.append(metrics)
        print(f"  SHD={metrics['SHD']}, TPR={metrics['TPR']:.3f}, FDR={metrics['FDR']:.3f}")

    except Exception as e:
        print(f"  ERROR on {rel}: {e}")
        
scores_df = pd.DataFrame(results)

  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
Processing: casual_effect/device_failure_data.csv


/dcs/23/u2200504/thesis/recidivism-causal/code/lingam/lingam/lim.py:119: RuntimeWarning: overflow encountered in cosh
  loss_con = np.sum(-np.log(np.cosh(R)) * dis_con)
/dcs/23/u2200504/thesis/envthesis/lib/python3.13/site-packages/numpy/linalg/linalg.py:680: RuntimeWarning: overflow encountered in matmul
  result = z if result is None else fmatmul(result, z)
/dcs/23/u2200504/thesis/recidivism-causal/code/lingam/lingam/lim.py:251: RuntimeWarning: invalid value encountered in multiply
  h = (G_h.T * M).sum() - d
/dcs/23/u2200504/thesis/recidivism-causal/code/lingam/lingam/lim.py:117: RuntimeWarning: invalid value encountered in logaddexp
  (np.logaddexp(0, M) - X * M) * np.absolute(dis_con - 1)
/dcs/23/u2200504/thesis/envthesis/lib/python3.13/site-packages/numpy/linalg/linalg.py:677: RuntimeWarning: overflow encountered in matmul
  z = a if z is None else fmatmul(z, z)
/dcs/23/u2200504/thesis/recidivism-causal/code/lingam/lingam/lim.py:263: RuntimeWarning: overflow encountered in scalar

W_est (without the 2nd phase) is: 
 [[ 0.          0.26656997  0.          0.55855538 -0.58457226  0.30952723
   0.13478376]
 [ 0.          0.          0.          0.17586074  0.27680767 -0.65267261
   0.        ]
 [ 0.          0.          0.          0.          0.          0.
   0.        ]
 [ 0.          0.          0.          0.         -0.56440679  0.
   0.        ]
 [ 0.          0.          0.          0.          0.          0.
   0.        ]
 [ 0.          0.          0.          0.          0.          0.
   0.        ]
 [ 0.         -0.19705668  0.40414248  0.2760606   0.         -0.19476431
   0.        ]]
  Saved graph to results/graphs_LiM/casual_effect__device_failure_data__LiM.png
metrics computed
{'TP': 5, 'FP': 8, 'FN': 4, 'TN': 32, 'SHD': 12, 'TPR': 0.5555555555555556, 'FPR': 0.2, 'FDR': 0.6153846153846154}
  SHD=12, TPR=0.556, FDR=0.615
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
Processing: casual_effect/stud

/dcs/23/u2200504/thesis/recidivism-causal/code/lingam/lingam/lim.py:119: RuntimeWarning: overflow encountered in cosh
  loss_con = np.sum(-np.log(np.cosh(R)) * dis_con)
/dcs/23/u2200504/thesis/recidivism-causal/code/lingam/lingam/lim.py:263: RuntimeWarning: overflow encountered in scalar multiply
  obj = loss + 0.5 * rho * h * h + alpha * h + self._lambda1 * w.sum()
/dcs/23/u2200504/thesis/recidivism-causal/code/lingam/lingam/lim.py:117: RuntimeWarning: invalid value encountered in logaddexp
  (np.logaddexp(0, M) - X * M) * np.absolute(dis_con - 1)
/dcs/23/u2200504/thesis/recidivism-causal/code/lingam/lingam/lim.py:265: RuntimeWarning: overflow encountered in multiply
  G_smooth = G_loss + (rho * h + alpha) * G_h.T * W * 2  # 2019


W_est (without the 2nd phase) is: 
 [[ 0.          0.53728677 -0.49246039  0.        ]
 [ 0.          0.          0.18444206 -0.24652095]
 [ 0.56044306  0.          0.         -0.52343207]
 [ 0.56550098  0.77455133  0.          0.        ]]
  Saved graph to results/graphs_LiM/causal_direction_iv__ecommerce_sem__LiM.png
metrics computed
{'TP': 3, 'FP': 5, 'FN': 0, 'TN': 8, 'SHD': 5, 'TPR': 1.0, 'FPR': 0.38461538461538464, 'FDR': 0.625}
  SHD=5, TPR=1.000, FDR=0.625
Processing: causal_direction_iv/environment_sem.csv


...

W_est (without the 2nd phase) is: 
 [[ 0.         -0.3333772   0.38665599  1.52598351]
 [ 0.          0.         -0.33673977  0.        ]
 [ 0.          0.          0.          0.        ]
 [ 0.          0.83321908  0.28137166  0.        ]]
  Saved graph to results/graphs_LiM/causal_direction_iv__environment_sem__LiM.png
metrics computed
{'TP': 2, 'FP': 4, 'FN': 1, 'TN': 9, 'SHD': 5, 'TPR': 0.6666666666666666, 'FPR': 0.3076923076923077, 'FDR': 0.6666666666666666}
  SHD=5, TPR=0.667, FDR=0.667
Processing: causal_direction_iv/marketing_sem.csv
W_est (without the 2nd phase) is: 
 [[ 0.          0.          0.          0.        ]
 [ 0.          0.         -2.15634265  2.07086142]
 [ 1.11243015  0.          0.          0.        ]
 [ 0.          0.          1.66356567  0.        ]]
  Saved graph to results/graphs_LiM/causal_direction_iv__marketing_sem__LiM.png
metrics computed
{'TP': 1, 'FP': 3, 'FN': 2, 'TN': 10, 'SHD': 5, 'TPR': 0.3333333333333333, 'FPR': 0.23076923076923078, 'FDR': 0.75

/dcs/23/u2200504/thesis/recidivism-causal/code/lingam/lingam/lim.py:119: RuntimeWarning: overflow encountered in cosh
  loss_con = np.sum(-np.log(np.cosh(R)) * dis_con)


W_est (without the 2nd phase) is: 
 [[ 0.          0.          0.93738358 -0.25814549]
 [ 0.33134036  0.          0.         -0.16848797]
 [ 0.          0.          0.          1.58155036]
 [ 0.          0.          0.          0.        ]]
  Saved graph to results/graphs_LiM/domain_shift__domain_shift_sem__LiM.png
metrics computed
{'TP': 3, 'FP': 2, 'FN': 2, 'TN': 9, 'SHD': 4, 'TPR': 0.6, 'FPR': 0.18181818181818182, 'FDR': 0.4}
  SHD=4, TPR=0.600, FDR=0.400
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
Processing: mediation_outcome_confounder/exercise_intervention_study.csv
W_est (without the 2nd phase) is: 
 [[ 0.          0.          0.          1.76800605  2.00157674  0.
   0.        ]
 [ 0.          0.         -0.90795236  0.          0.60724245 -0.51424379
   0.        ]
 [ 0.          0.          0.          0.          0.          0.
   0.        ]
 [ 0.       

/dcs/23/u2200504/thesis/recidivism-causal/code/lingam/lingam/lim.py:119: RuntimeWarning: overflow encountered in cosh
  loss_con = np.sum(-np.log(np.cosh(R)) * dis_con)


W_est (without the 2nd phase) is: 
 [[0.         0.         0.         0.        ]
 [0.         0.         0.         0.87772109]
 [0.         0.         0.         0.        ]
 [0.42742395 0.         0.40709761 0.        ]]
  Saved graph to results/graphs_LiM/necessity_sufficiency__bridge_integrity__LiM.png
metrics computed
{'TP': 1, 'FP': 2, 'FN': 2, 'TN': 11, 'SHD': 4, 'TPR': 0.3333333333333333, 'FPR': 0.15384615384615385, 'FDR': 0.6666666666666666}
  SHD=4, TPR=0.333, FDR=0.667
Processing: necessity_sufficiency/factory_monitoring.csv
W_est (without the 2nd phase) is: 
 [[0.         0.         0.         0.        ]
 [0.         0.         0.         0.        ]
 [0.         0.         0.         0.83632765]
 [0.42264657 0.44368063 0.         0.        ]]
  Saved graph to results/graphs_LiM/necessity_sufficiency__factory_monitoring__LiM.png
metrics computed
{'TP': 1, 'FP': 2, 'FN': 2, 'TN': 11, 'SHD': 4, 'TPR': 0.3333333333333333, 'FPR': 0.15384615384615385, 'FDR': 0.666666666666666

In [9]:
scores_df

,TP,FP,FN,TN,SHD,TPR,FPR,FDR,scenario,dataset,algo
0,5,8,4,32,12,0.555556,0.200000,0.615385,casual_effect,device_failure_data,lim
1,4,5,5,35,10,0.444444,0.125000,0.555556,casual_effect,student_tutoring_data,lim
2,2,2,1,11,3,0.666667,0.153846,0.500000,causal_direction_iv,clinical_trial_sem,lim
3,3,5,0,8,5,1.000000,0.384615,0.625000,causal_direction_iv,ecommerce_sem,lim
4,2,4,1,9,5,0.666667,0.307692,0.666667,causal_direction_iv,environment_sem,lim
5,1,3,2,10,5,0.333333,0.230769,0.750000,causal_direction_iv,marketing_sem,lim
6,2,2,2,10,4,0.500000,0.166667,0.500000,counterfactual_reasoning,climate_impact_sem,lim
7,1,3,3,9,6,0.250000,0.250000,0.750000,counterfactual_reasoning,clinical_trial_sem,lim
8,1,3,3,9,6,0.250000,0.250000,0.750000,counterfactual_reasoning,education_performance_sem,lim
9,1,4,3,8,7,0.250000,0.333333,0.800000,counterfactual_reasoning,investment_outcome_sem,lim


In [10]:
scores_path = project_root / "results" / "scores"/"scores_lim.csv"
scores_path.parent.mkdir(parents=True, exist_ok=True)
scores_df.to_csv(scores_path, index=False)